In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "earthaccess-auth[icechunk]>=0.4.0",
#     "icechunk>=2.2.2",
#     "matplotlib>=3.11.2",
#     "xarray>=2026.7.0",
# ]
# ///

# Quickstart: the ASDC bucket snapshot

NASA ASDC hosts an experimental copy of each store in its public bucket,
`s3://asdc-prod-public/virtual-icechunk/`, as `TEMPO_HCHO_L3_V04` and
`TEMPO_NO2_L3_V04`. Each is a snapshot of the
[Source Cooperative](https://source.coop/pangeo/tempo-virtual-icechunk)
copy at the time it was made, not a live store. Expect it to lag the
private store that the pipeline updates, and expect it to move or change
without notice.

The bucket is public, so opening the repository needs no credentials. This
notebook opens it through ASDC's HTTPS endpoint, which serves the bucket's
keys to anyone and so works from anywhere. The data is still *virtual*:
each chunk is a byte range in a granule in `s3://asdc-prod-protected/`,
which needs an [Earthdata Login](https://urs.earthdata.nasa.gov/) and a
machine in AWS us-west-2, exactly as in the
[first quickstart](../quickstart/). This snapshot stores its references as
absolute `s3://` URLs rather than relative ones, so the HTTPS trick of the
[second quickstart](../quickstart-https/) does not apply to it.

This notebook reads one point of the HCHO store. For NO2, swap `HCHO` for
`NO2` in the URL below and pick a variable from
[the stores](../../reference/stores/). Run it with
[juv](https://github.com/manzt/juv): `juv run docs/examples/quickstart-asdc.ipynb`.

## Authenticate

`login()` looks for credentials in `EARTHDATA_USERNAME` and
`EARTHDATA_PASSWORD` (or `EARTHDATA_TOKEN`), then `~/.netrc`, then prompts.
Only the chunk reads below fetch S3 credentials through this login, when
they need them, and refresh them before they expire.


In [ ]:
import earthaccess_auth

earthaccess_auth.set_default_auth(earthaccess_auth.login())

## Open the store

`http_storage` reads the repository over HTTPS with no credentials. The
endpoint answers each request with a redirect to a signed CloudFront URL,
which Icechunk follows. Give it the repository root without a trailing
slash. The repository's config declares which bucket its virtual chunks
live in. `earthdata_containers_credentials` reads that config from the open
repository and authorizes Icechunk to fetch chunks from the bucket with
Earthdata credentials. `reopen` applies them without reading the config
from storage again. xarray opens the `main` branch as one dataset.


In [ ]:
import icechunk
import xarray as xr
from earthaccess_auth.adapters.icechunk import earthdata_containers_credentials

storage = icechunk.http_storage(
    "https://data.asdc.earthdata.nasa.gov/asdc-prod-public/virtual-icechunk/TEMPO_HCHO_L3_V04"
)
repo = icechunk.Repository.open(storage)
repo = repo.reopen(
    authorize_virtual_chunk_access=earthdata_containers_credentials(repo)
)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks=None)
ds

## Load a single point

Nothing has been read yet beyond metadata and the coordinates. Selecting a
point and a time, then asking for the value, reads the one chunk that holds
it, from `asdc-prod-protected`. `main_data_quality_flag` is 0 for normal
retrievals. The [first quickstart](../quickstart/) goes on to plot a map
and a timeseries; every cell there works unchanged on this `ds`.


In [ ]:
nyc = dict(latitude=40.7, longitude=-74.0, method="nearest")
scan = ds.sel(time="2026-09-03T18:00", method="nearest")

hcho = scan.vertical_column.sel(**nyc)
flag = scan.main_data_quality_flag.sel(**nyc)
when = scan.time.values.astype("datetime64[m]")
print(f"{when}: {float(hcho):.3g} {hcho.units}, quality flag {int(flag)}")